# 🤖 AI Feedback Learning Platform — DPO Fine-Tuning

**Purpose:** Fine-tune a small open-source LLM using Direct Preference Optimization (DPO)  
based on user feedback collected by the platform.  
**Cost:** $0 — runs entirely on Google Colab's free T4 GPU.  
**Output:** A LoRA-adapted model pushed to your Hugging Face Hub.

---
### 📋 Before You Start
1. **Runtime → Change runtime type → T4 GPU** (free in Colab)
2. Upload your `dpo_dataset.jsonl` file (generated by `prepare_dataset.py`)
3. Set your secrets in the form below


In [ ]:
# @title ⚙️ Configuration — Fill These In
HF_TOKEN        = ""   # @param {type:"string"} Your Hugging Face write token
HF_MODEL_REPO   = "your-username/ai-feedback-model"  # @param {type:"string"}
BASE_MODEL_ID   = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # @param {type:"string"} Free, small, fast
DATASET_FILE    = "dpo_dataset.jsonl"  # @param {type:"string"}
MAX_STEPS       = 200   # @param {type:"integer"} Keep low for Colab free tier
LEARNING_RATE   = 5e-5  # @param {type:"number"}
BETA            = 0.1   # @param {type:"number"} DPO temperature — controls how strongly to prefer 'chosen'

In [ ]:
# @title 📦 Step 1: Install Dependencies
!pip install -q \
    transformers==4.41.2 \
    trl==0.9.4 \
    peft==0.11.1 \
    accelerate==0.31.0 \
    bitsandbytes==0.43.1 \
    datasets==2.19.1 \
    huggingface_hub
print('✅ Dependencies installed.')

In [ ]:
# @title 🔐 Step 2: Authenticate with Hugging Face
from huggingface_hub import login
login(token=HF_TOKEN)
print('✅ Logged in to Hugging Face Hub.')

In [ ]:
# @title 🔍 Step 3: Verify GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Running on: {device.upper()}')
if device == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU.')

In [ ]:
# @title 📂 Step 4: Load and Validate the DPO Dataset
from datasets import load_dataset
import json

dataset = load_dataset('json', data_files=DATASET_FILE, split='train')
print(f'✅ Loaded {len(dataset)} preference pairs.')
print(f'\n📌 Sample pair:')
sample = dataset[0]
print(f'  PROMPT:   {sample["prompt"][:120]}...')
print(f'  CHOSEN:   {sample["chosen"][:120]}...')
print(f'  REJECTED: {sample["rejected"][:120]}...')

# Split 90% train / 10% eval
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset  = split['test']
print(f'\n  Train: {len(train_dataset)} | Eval: {len(eval_dataset)}')

In [ ]:
# @title 🏗️ Step 5: Load Base Model with 4-bit Quantization (fits in free T4 VRAM)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f'⏳ Loading base model: {BASE_MODEL_ID} ...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'✅ Model loaded. Parameters: {model.num_parameters()/1e6:.0f}M')

In [ ]:
# @title 🎯 Step 6: Configure LoRA Adapters (Parameter-Efficient Fine-Tuning)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,                   # Rank — higher = more capacity, more memory
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
trainable, total = model.get_nb_trainable_parameters()
print(f'✅ LoRA applied. Trainable params: {trainable/1e6:.2f}M / {total/1e6:.0f}M ({100*trainable/total:.2f}%)')

In [ ]:
# @title 🚀 Step 7: DPO Training
from trl import DPOTrainer, DPOConfig

dpo_config = DPOConfig(
    beta=BETA,
    output_dir='./dpo_output',
    max_steps=MAX_STEPS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    logging_steps=10,
    evaluation_strategy='steps',
    eval_steps=50,
    save_steps=100,
    fp16=True,           # Use mixed precision for T4
    report_to='none',    # Disable wandb to keep it free
    remove_unused_columns=False,
    max_length=512,
    max_prompt_length=256,
)

trainer = DPOTrainer(
    model=model,
    args=dpo_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
)

print('🏋️  Starting DPO training...')
trainer.train()
print('✅ Training complete!')

In [ ]:
# @title 📊 Step 8: Evaluate — Before vs After
model.eval()

test_prompt = train_dataset[0]['prompt']
inputs = tokenizer(test_prompt, return_tensors='pt').to(device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)

generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print(f'📌 Test Prompt:  {test_prompt[:200]}')
print(f'\n🤖 Fine-tuned Response:')
print(generated)
print(f'\n✅ Chosen (ground truth): {train_dataset[0]["chosen"][:200]}')

In [ ]:
# @title ☁️ Step 9: Push Fine-Tuned Model to Hugging Face Hub
print(f'⏳ Pushing model to: {HF_MODEL_REPO} ...')
trainer.model.push_to_hub(HF_MODEL_REPO, private=True)
tokenizer.push_to_hub(HF_MODEL_REPO, private=True)

print(f'\n✅ Model pushed successfully!')
print(f'   👉 https://huggingface.co/{HF_MODEL_REPO}')
print(f'\n🎉 Next: Update your Inference Lambda to use this new model.')

## ✅ Training Complete!

Your fine-tuned model (with LoRA adapters) is now on Hugging Face Hub.

**Next steps:**
1. Update the `MODEL_ID` environment variable in your Inference Lambda (or switch to the HF Serverless API)
2. The GitHub Actions workflow (`retrain.yml`) will automate this entire notebook run on a schedule
3. Check your Streamlit dashboard for improved satisfaction scores over time
